# Sales Lakehouse Setup

## Objective

Create the Unity Catalog schemas and control tables required by the Sales lakehouse project.

The catalog is supplied as a runtime parameter so the same setup can be reused across environments.


In [0]:
# ============================================================
# 1. CREATE SETUP PARAMETER
# ============================================================

dbutils.widgets.text("catalog_name", "workspace")
catalog_name = dbutils.widgets.get("catalog_name").strip()

if not catalog_name:
    raise ValueError("catalog_name cannot be empty")

print(f"Catalog: {catalog_name}")


In [0]:
# ============================================================
# 2. CREATE SALES SCHEMAS
# ============================================================

schemas = [
    "sales_bronze",
    "sales_silver",
    "sales_gold",
    "sales_quarantine",
    "sales_metadata",
]

for schema_name in schemas:
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}"
    )
    print(f"Ensured schema exists: {catalog_name}.{schema_name}")


In [0]:
# ============================================================
# 3. CREATE INGESTION WATERMARK CONTROL TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.sales_metadata.ingestion_watermark (
    entity STRING NOT NULL,
    watermark_column STRING NOT NULL,
    last_successful_watermark TIMESTAMP,
    updated_at TIMESTAMP NOT NULL
)
USING DELTA
""")

print(
    f"Ensured control table exists: "
    f"{catalog_name}.sales_metadata.ingestion_watermark"
)


## Schemas

- `sales_bronze` — raw or minimally transformed Sales data
- `sales_silver` — cleaned, standardized, validated, and deduplicated data
- `sales_gold` — business-ready facts, dimensions, and aggregates
- `sales_quarantine` — rejected records that fail data-quality checks
- `sales_metadata` — pipeline state, audit results, and operational metadata

## Control Table

`sales_metadata.ingestion_watermark` stores the last successfully committed watermark for incremental entities. The watermark must only advance after the corresponding ingestion run succeeds and is validated.
